In [1]:
import re
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# ─────────────────────────────────────────────────────────────
# 1. DEFINICIÓN DE SECTORES Y SUS PALABRAS CLAVE
# ─────────────────────────────────────────────────────────────

SECTORES = {
    "Salud": [
        "salud", "hospital", "clínica", "médico", "farmacéutico", "medicamento",
        "medicina", "enfermería", "essalud", "minsa", "vacuna", "epidemia",
        "pandemia", "covid", "enfermedad", "paciente", "sis", "salud mental",
        "cáncer", "rehabilitación", "emergencia sanitaria", "atención médica"
    ],
    "Educación": [
        "educación", "docente", "estudiante", "escuela", "colegio",
        "universidad", "instituto", "enseñanza", "aprendizaje", "becas",
        "pronabec", "sunedu", "currículo", "educación superior",
        "educación básica", "infraestructura educativa"
    ],
    "Transporte e Infraestructura Vial": [
        "transporte", "carretera", "vía", "pista", "puente",
        "metro", "tren", "ferroviario", "aeropuerto", "puerto",
        "mtc", "tránsito", "seguridad vial"
    ],
    "Vivienda, Construcción y Saneamiento": [
        "vivienda", "construcción", "obra pública", "urbanismo",
        "agua potable", "alcantarillado", "desagüe", "saneamiento",
        "mvcs", "cofopri", "habilitación urbana"
    ],
    "Seguridad y Defensa": [
        "seguridad ciudadana", "policía", "pnp", "militar",
        "ejército", "marina", "fuerza aérea", "fuerzas armadas",
        "mindef", "terrorismo", "tráfico de drogas", "orden público",
        "serenazgo"
    ],
    "Energía": [
        "energía", "electricidad", "electrificación",
        "energía renovable", "paneles solares", "osinergmin"
    ],
    "Minas": [
        "minería", "minero", "concesión minera",
        "hidrocarburos", "petróleo", "gas natural", "perupetro"
    ],
    "Ambiental": [
        "medio ambiente", "ambiental", "minam", "bosques",
        "biodiversidad", "contaminación", "cambio climático",
        "residuos sólidos", "oefa", "reforestación", "deforestación"
    ],
    "Tecnología": [
        "tecnología", "digital", "innovación", "software",
        "inteligencia artificial", "datos", "internet",
        "ciberseguridad", "transformación digital", "gobierno digital"
    ],
    "Justicia": [
        "penal", "delito", "pena", "corrupción",
        "poder judicial", "fiscalía", "tribunal",
        "juzgado", "derechos humanos", "violencia",
        "debido proceso", "inpe"
    ],
    "Cultura": [
        "cultura", "patrimonio cultural", "museo",
        "arqueología", "historia", "arte",
        "cine", "teatro", "lenguas indígenas"
    ],
    "Deporte": [
        "deporte", "deportivo", "fútbol", "juegos",
        "olimpiadas", "ipd", "estadio",
        "actividad física", "recreación"
    ],
    "Comercio y Alimentación": [
        "comercio", "mercado", "alimentos", "alimentación",
        "exportación", "importación", "industria",
        "mipyme", "empresa", "indecopi"
    ],
    "Desastres y Gestión de Riesgo": [
        "desastre", "emergencia", "riesgo", "indeci",
        "inundación", "sismo", "terremoto", "huayco",
        "sequía", "incendio", "prevención", "reconstrucción"
    ],
    "Agricultura y Ganadería": [
        "agricultura", "agrario", "ganadería", "pecuario",
        "midagri", "riego", "fertilizantes", "senasa",
        "agroindustria", "productores agropecuarios"
    ],
    "Administración Pública": [
        "administración pública", "estado", "gobierno",
        "congreso", "pcm", "función pública",
        "servir", "contraloría", "gestión pública",
        "descentralización"
    ],
}

# ─────────────────────────────────────────────────────────────
# 2. PREPROCESAMIENTO DE TEXTO
# ─────────────────────────────────────────────────────────────

def limpiar_texto(texto):
    """Minúsculas, sin acentos, sin caracteres especiales, sin stopwords."""
    texto = texto.lower()
    reemplazos = {'á':'a','é':'e','í':'i','ó':'o','ú':'u','ü':'u','ñ':'n'}
    for orig, remp in reemplazos.items():
        texto = texto.replace(orig, remp)
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    stopwords = {
        'de','la','el','en','y','a','del','al','por','para','con','los','las',
        'un','una','unos','unas','se','su','sus','es','son','como','mas','menos',
        'o','que','lo','le','les','nos','si','e','u','fue','ser','ha','han',
    }
    return ' '.join(p for p in texto.split() if p not in stopwords)


def limpiar_denominacion(texto):
    texto = str(texto).upper()

    patrones = [
        r'LEY QUE', r'LEY DE', r'LEY Nº \d+',
        r'RESOLUCION LEGISLATIVA',
        r'EL CONGRESO DE LA REPUBLICA',
        r'DE LA REPUBLICA',
        r'SE MODIFICA', r'SE APRUEBA'
    ]

    for p in patrones:
        texto = re.sub(p, ' ', texto)

    return re.sub(r'\s+', ' ', texto).strip()


SECTORES_NORM = {
    s: [limpiar_texto(k) for k in kws]
    for s, kws in SECTORES.items()
}

# ─────────────────────────────────────────────────────────────
# 3. REGLAS DE ALTA PRECISIÓN PARA LEYES
# ─────────────────────────────────────────────────────────────

def clasificar_reglas(texto, texto_original): 
    orig = texto_original.upper() 
    # Salud 
    if any(p in orig for p in ['EMERGENCIA SANITARIA', 'COVID', 'CORONAVIRUS']): 
        return 'Salud' 
    # Educación 
    if any(p in orig for p in ['LEY UNIVERSITARIA', 'SUNEDU', 'UNIVERSIDAD']): 
        return 'Educación' 
    # Seguridad 
    if 'INGRESO DE' in orig and ('MILITAR' in orig or 'NAVAL' in orig): 
        return 'Seguridad y Defensa' 
    # Justicia 
    if any(p in orig for p in ['CODIGO PENAL', 'CRIMEN ORGANIZADO']): 
        return 'Justicia' 
    # Transporte 
    if any(p in orig for p in ['CARRETERA', 'TRANSPORTE', 'MTC']): 
        return 'Transporte e Infraestructura Vial' 
    # Vivienda 
    if any(p in orig for p in ['SANEAMIENTO', 'AGUA POTABLE', 'ALCANTARILLADO']): 
        return 'Vivienda, Construcción y Saneamiento' 
    # Desastres 
    if any(p in orig for p in ['FENOMENO EL NINO', 'RECONSTRUCCION', 'DESASTRE']): 
        return 'Desastres y Gestión de Riesgo'
    
    if any(p in orig for p in [
    'MODIFICA LA LEY', 'CREA EL SISTEMA', 'FORTALECE LA GESTION',
    'ORGANISMO', 'ENTIDAD PUBLICA']):
        return 'Administración Pública'

    # Economía / comercio
    if any(p in orig for p in ['IMPUESTO', 'TRIBUTARIO', 'SUNAT', 'ARANCEL']):
        return 'Comercio y Alimentación'

    # Trabajo
    if any(p in orig for p in ['TRABAJADOR', 'EMPLEO', 'REMUNERACION', 'LABORAL']):
        return 'Administración Pública'
    
    return None 

# ─────────────────────────────────────────────────────────────
# 4. CLASIFICADOR TF-IDF
# ─────────────────────────────────────────────────────────────

class ClasificadorLeyes:
    """
    Pipeline: Regla → TF-IDF coseno sobre perfiles de sector.
    """
    def __init__(self):
        self.sector_names = list(SECTORES_NORM.keys())
        perfiles = [' '.join(kws) for kws in SECTORES_NORM.values()]
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
        self.vectorizer.fit(perfiles)
        self.perfil_matrix = self.vectorizer.transform(perfiles)

    def clasificar(self, denominacion):
        denominacion = str(denominacion)
        texto_limpio_orig = limpiar_denominacion(denominacion)
        texto = limpiar_texto(texto_limpio_orig)

        # 1) Reglas primero
        sector_regla = clasificar_reglas(texto, denominacion)
        if sector_regla is not None:
            return sector_regla, 'Regla'

        # 2) TF-IDF coseno
        vec = self.vectorizer.transform([texto])
        sims = cosine_similarity(vec, self.perfil_matrix)[0]
        idx = int(np.argmax(sims))

        if sims[idx] < 0.04:
            return 'Sin clasificar', 'Ninguno'

        return self.sector_names[idx], 'TF-IDF'

# ─────────────────────────────────────────────────────────────
# 5. PIPELINE PRINCIPAL
# ─────────────────────────────────────────────────────────────

print('=' * 60)
print('CLASIFICADOR NLP DE LEYES DEL CONGRESO')
print('=' * 60)

df = pd.read_csv('leyes.csv')
print(f'\nLeyes a clasificar: {len(df)}\n')

df_res = df.copy()
clasificador = ClasificadorLeyes()

sectores, metodos = [], []

print('RESULTADOS:')
for i, denominacion in enumerate(df_res['Descripción']):
    sector, metodo = clasificador.clasificar(denominacion)
    sectores.append(sector)
    metodos.append(metodo)
    print(f'[{i+1}/{len(df_res)}]')
    print(f'Descripción : {str(denominacion)[:80]}...')
    print(f'Sector       : {sector}')
    print(f'Método       : {metodo}')
    print()

df_res['sector_clasificado'] = sectores
df_res['metodo_clasificacion'] = metodos

output_path = 'leyes_clasificadas.xlsx'
df_res.to_excel(output_path, index=False)
print(f'Resultados exportados a: {output_path}')

CLASIFICADOR NLP DE LEYES DEL CONGRESO

Leyes a clasificar: 1454

RESULTADOS:
[1/1454]
Descripción : LEY QUE INCORPORA LA TERCERA DISPOSICIÓN TRANSITORIA DE LA LEY 26859, LEY ORGÁNI...
Sector       : Sin clasificar
Método       : Ninguno

[2/1454]
Descripción : LEY QUE DELEGA EN EL PODER EJECUTIVO LA FACULTAD DE LEGISLAR EN DIVERSAS MATERIA...
Sector       : Salud
Método       : Regla

[3/1454]
Descripción : LEY DE PROTECCIÓN POLICIAL...
Sector       : Sin clasificar
Método       : Ninguno

[4/1454]
Descripción : LEY QUE MODIFICA EL ARTÍCULO 34 DE LA LEY 29459, LEY DE LOS PRODUCTOS FARMACÉUTI...
Sector       : Sin clasificar
Método       : Ninguno

[5/1454]
Descripción : LEY QUE REGULA EL USO DE SUSTANCIAS MODELANTES EN TRATAMIENTOS CORPORALES CON FI...
Sector       : Salud
Método       : TF-IDF

[6/1454]
Descripción : LEY QUE AUTORIZA LA EJECUCIÓN DE INTERVENCIONES EN INFRAESTRUCTURA SOCIAL BÁSICA...
Sector       : Educación
Método       : TF-IDF

[7/1454]
Descripción : LEY QUE ESTABL

In [2]:
conteo = df_res['sector_clasificado'].value_counts()
porcentaje = df_res['sector_clasificado'].value_counts(normalize=True) * 100

print('\nDistribución de leyes por sector:')
print('-' * 55)
for sector in conteo.index:
    print(f'{sector:<42} {conteo[sector]:>4}  ({porcentaje[sector]:.1f}%)')
print('-' * 55)
print(f'Total: {len(df_res)}')


Distribución de leyes por sector:
-------------------------------------------------------
Administración Pública                      400  (27.5%)
Sin clasificar                              370  (25.4%)
Educación                                   117  (8.0%)
Seguridad y Defensa                         109  (7.5%)
Vivienda, Construcción y Saneamiento         97  (6.7%)
Salud                                        86  (5.9%)
Justicia                                     66  (4.5%)
Transporte e Infraestructura Vial            59  (4.1%)
Comercio y Alimentación                      41  (2.8%)
Cultura                                      29  (2.0%)
Tecnología                                   19  (1.3%)
Desastres y Gestión de Riesgo                18  (1.2%)
Agricultura y Ganadería                      15  (1.0%)
Ambiental                                    11  (0.8%)
Deporte                                      11  (0.8%)
Minas                                         5  (0.3%)
Energía    

In [3]:
df_res['metodo_clasificacion'].value_counts()

TF-IDF     603
Regla      481
Ninguno    370
Name: metodo_clasificacion, dtype: int64